## 1. Pure Matchers (scoring_metrics.py)

Simple functions that take (response, ground_truth) and return (0 or 1, confidence).

In [1]:
from Evaluation.scoring_metrics import MultipleChoiceMatcher, MathMatcher, BehaviorMatcher, SemanticMatcher

print("Loaded all matchers!")

Loaded all matchers!


### 1.1 MathMatcher (GMS8K)

In [5]:
matcher = MathMatcher()

# Test cases
test_cases = [
    ("The answer is 42", "#### 42"),
    ("After calculation, we get 100", "#### 100"),
    ("I think it's about 50", "#### 42"),
    ("3 + 5 = 8. So the answer is 8.", "#### 8"),
]

print("MathMatcher Tests:")
print("=" * 60)
for response, gt in test_cases:
    is_correct, conf = matcher.check(response, gt)
    status = "✓" if is_correct else "✗"
    print(f"{status} Response: '{response[:30]}...' | GT: {gt} -> ({is_correct}, {conf:.2f})")

MathMatcher Tests:
✓ Response: 'The answer is 42...' | GT: #### 42 -> (1, 1.00)
✓ Response: 'After calculation, we get 100...' | GT: #### 100 -> (1, 1.00)
✗ Response: 'I think it's about 50...' | GT: #### 42 -> (0, 0.00)
✓ Response: '3 + 5 = 8. So the answer is 8....' | GT: #### 8 -> (1, 1.00)


### 1.2 MultipleChoiceMatcher (CSQA)

In [6]:
matcher = MultipleChoiceMatcher(device="cuda:0")

choices = {
    "A": "race track",
    "B": "populated areas",
    "C": "the desert",
    "D": "apartment",
    "E": "roadblock",
}
correct_label = "B"

test_responses = [
    "The answer is B",
    "B. populated areas",
    "I think populated areas is correct",
    "A - race track",
    "The desert seems right",
]

print("MultipleChoiceMatcher Tests:")
print("=" * 60)
for response in test_responses:
    is_correct, conf = matcher.check(response, choices, correct_label)
    status = "✓" if is_correct else "✗"
    print(f"{status} '{response}' -> ({is_correct}, {conf:.2f})")

MultipleChoiceMatcher Tests:
✓ 'The answer is B' -> (1, 1.00)
✓ 'B. populated areas' -> (1, 1.00)
✓ 'I think populated areas is correct' -> (1, 0.90)
✗ 'A - race track' -> (0, 0.00)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✗ 'The desert seems right' -> (0, 0.75)


### 1.3 BehaviorMatcher

In [8]:
# Keyword-only mode (fast)
detector = BehaviorMatcher(device="cuda:0")

test_responses = [
    "I cannot help with that request.",
    "The capital of France is Paris.",
    "I'm sorry, but I can't assist with harmful content.",
    "Sure! Here's how to solve the problem...",
    "As an AI, I don't have personal opinions.",
]

print("BehaviorMatcher Tests (keyword mode):")
print("=" * 60)
for response in test_responses:
    is_refusal, conf = detector.check(response)
    status = "REFUSE" if is_refusal else "NORMAL"
    print(f"[{status:6s}] ({conf:.2f}) {response[:50]}")

BehaviorMatcher Tests (keyword mode):


Device set to use cuda:0


[REFUSE] (0.99) I cannot help with that request.
[NORMAL] (0.00) The capital of France is Paris.
[REFUSE] (0.98) I'm sorry, but I can't assist with harmful content
[NORMAL] (0.00) Sure! Here's how to solve the problem...
[REFUSE] (0.51) As an AI, I don't have personal opinions.


### 1.4 SemanticMatcher

In [9]:
matcher = SemanticMatcher(device="cuda:0", threshold=0.7)

test_cases = [
    ("Paris is the capital of France", "Paris"),
    ("The answer is approximately forty-two", "42"),
    ("Dogs are loyal pets", "Cats are independent"),
]

print("SemanticMatcher Tests:")
print("=" * 60)
for response, gt in test_cases:
    is_match, similarity = matcher.check(response, gt)
    status = "✓" if is_match else "✗"
    print(f"{status} Response: '{response[:30]}' | GT: '{gt}' -> sim={similarity:.2f}")

SemanticMatcher Tests:
✓ Response: 'Paris is the capital of France' | GT: 'Paris' -> sim=1.00
✗ Response: 'The answer is approximately fo' | GT: '42' -> sim=0.52
✗ Response: 'Dogs are loyal pets' | GT: 'Cats are independent' -> sim=0.46


## 2. Convenience Functions

In [ ]:
from Evaluation import check_math_answer, check_behavior, check_semantic

# One-liner usage
print("One-liner convenience functions:")
print(check_math_answer("The answer is 42", "#### 42"))
print(check_behavior("I cannot do that", use_nli=False))
print(check_semantic("Paris is beautiful", "Paris"))

## 3. Full Evaluation Pipeline (eval_pipeline.py)

Orchestrates loading, steering, and evaluation.

In [3]:
from Evaluation.eval_pipeline import EvalPipeline

pipeline = EvalPipeline(
    model_name="google/gemma-2-2b",
    device="cuda:0",
)

print(f"Supported methods: {pipeline.SUPPORTED_METHODS}")
print(f"Supported datasets: {pipeline.SUPPORTED_TEST_DATASETS}")

Supported methods: ['CAA', 'SRPS', 'CAST']
Supported datasets: ['csqa', 'gms8k', 'refusal']


In [ ]:

result = pipeline.evaluate(
    method="CAA",
    dataset="csqa",
    layer=12,
    coeff=1.0,
    n_samples=20,  # Small for demo
    n_train=50,
    include_baseline=True,
    verbose=True,
)


Evaluating CAA on csqa
Model: google/gemma-2-2b, Layer: 12, Coeff: 1.0


ModuleNotFoundError: No module named 'steering'

In [ ]:
# View results
print("\nResult Summary:")
print(f"  Accuracy:  {result.accuracy:.2%}")
print(f"  Baseline:  {result.baseline_accuracy:.2%}")
print(f"  Delta:     {result.accuracy - result.baseline_accuracy:+.2%}")
print(f"  Samples:   {result.total}")

In [ ]:
# Save results
from Evaluation import save_results

save_results(result, "./eval_results", output_format="both")

## 4. CLI Usage

```bash
# Evaluate CAA on CSQA
python Evaluation/eval_pipeline.py -m CAA -d csqa -n 100

# Evaluate CAST on GMS8K
python Evaluation/eval_pipeline.py -m CAST -d gms8k --layer 15 --coeff 2.0

# Evaluate all methods on all datasets
python Evaluation/eval_pipeline.py -m all -d all --output results/

# Quick test without baseline
python Evaluation/eval_pipeline.py -m CAA -d csqa -n 10 --no_baseline
```

## Summary

| Component | Purpose | Returns |
|-----------|---------|--------|
| `MultipleChoiceMatcher` | CSQA-style QA | (0/1, confidence) |
| `MathMatcher` | GMS8K math reasoning | (0/1, confidence) |
| `BehaviorMatcher` | Detect refusals | (0/1, confidence) |
| `SemanticMatcher` | General semantic match | (0/1, similarity) |
| `EvalPipeline` | Full orchestration | EvalResult |